# Module 03 — Reward Modeling

**Goal**: Train a model that scores responses from 0 to 1 — reflecting how "good" a response is.

This is the prerequisite for RLHF (Module 04). The reward model replaces a human labeller.

**What you will build**:
1. Format preference data (chosen / rejected pairs)
2. Add a scalar head to a language model
3. Train with `RewardTrainer`
4. Score responses and verify the ranking makes sense

In [ ]:
# Cell 1 — Install
!pip install -q transformers datasets trl peft accelerate torch

## 1. What is a reward model?

In RLHF, human labellers compare pairs of model responses and pick the better one.

A **reward model** learns to predict that human preference:

```
Input:  [prompt] + [response]
Output: scalar score  (higher = better)
```

Architecture: take a causal LM, remove the language modelling head, add a single linear
layer that maps the last hidden state to one scalar.

Training objective:

```
Loss = -log σ(r(x, y_w) - r(x, y_l))
        ↑ chosen response should score higher than rejected
```

This is a *Bradley-Terry* ranking loss — it only cares about the difference, not absolute values.

In [ ]:
# Cell 2 — Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from trl import RewardTrainer, RewardConfig
from peft import LoraConfig, TaskType

MODEL_ID  = "HuggingFaceTB/SmolLM2-135M"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE     = torch.bfloat16
MAX_STEPS = 80
print(f"Device: {DEVICE}")

## 2. Preference dataset

Format: each example has a `chosen` and a `rejected` response to the *same prompt*.
Chosen = the human preferred response.

In [ ]:
# Cell 3 — Build preference dataset
PREFS = [
    {
        "prompt": "What is gradient descent?",
        "chosen": "Gradient descent is an optimisation algorithm that minimises a loss function by repeatedly moving in the direction of the steepest downhill gradient. At each step, we compute the gradient and subtract a fraction of it (the learning rate) from the parameters.",
        "rejected": "Gradient descent is a thing in math. It goes down. People use it a lot.",
    },
    {
        "prompt": "Explain overfitting.",
        "chosen": "Overfitting occurs when a model learns the training data too well, capturing noise rather than the underlying pattern. This results in low training loss but high validation loss. Regularisation, dropout, early stopping, or gathering more data can help.",
        "rejected": "Overfitting means the model is bad at new data. You should get more data or something.",
    },
    {
        "prompt": "What is a transformer?",
        "chosen": "A transformer is a neural network architecture built around the self-attention mechanism. It processes all tokens in parallel rather than sequentially, allowing it to capture long-range dependencies efficiently. Transformers are the foundation of modern LLMs like GPT and Gemma.",
        "rejected": "Transformers are robots that turn into cars. In AI it's some kind of neural network.",
    },
    {
        "prompt": "Write a Python function to check if a number is prime.",
        "chosen": "def is_prime(n):\n    if n < 2: return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0: return False\n    return True",
        "rejected": "def is_prime(n):\n    for i in range(2, n):\n        if n % i == 0:\n            return False\n    return True  # slow O(n) version",
    },
    {
        "prompt": "What is the difference between precision and recall?",
        "chosen": "Precision is the fraction of predicted positives that are truly positive: TP/(TP+FP). Recall is the fraction of actual positives that were detected: TP/(TP+FN). High precision = few false alarms. High recall = few missed positives. The F1 score is their harmonic mean.",
        "rejected": "Precision means being precise and recall means remembering things. In ML they're both important metrics.",
    },
    {
        "prompt": "What is batch normalisation?",
        "chosen": "Batch normalisation normalises the activations of each layer to have zero mean and unit variance across the current mini-batch, then applies learnable scale and shift parameters. This stabilises training, allows higher learning rates, and reduces sensitivity to weight initialisation.",
        "rejected": "Batch norm makes training faster. It normalises stuff in batches.",
    },
    {
        "prompt": "Explain the vanishing gradient problem.",
        "chosen": "In deep networks, gradients are back-propagated by multiplying repeated Jacobians. When those values are < 1, the gradient shrinks exponentially with depth, making early layers learn extremely slowly or not at all. Solutions include residual connections, LSTM/GRU gates, and careful weight initialisation.",
        "rejected": "The vanishing gradient is when gradients become very small and learning stops. It's a problem in deep networks.",
    },
    {
        "prompt": "What is weight decay?",
        "chosen": "Weight decay (L2 regularisation) adds a penalty proportional to the sum of squared weights to the loss: L_total = L_task + λ·Σw². This discourages large weights, reducing overfitting and improving generalisation. In AdamW, weight decay is applied directly to the weights rather than through the gradient.",
        "rejected": "Weight decay makes weights smaller. It's a regularisation trick.",
    },
]

# Duplicate for more examples
PREFS = PREFS * 6   # 48 examples

# Format: RewardTrainer expects "input_ids_chosen", "attention_mask_chosen", etc.
# We'll pass raw text and let RewardTrainer handle tokenisation via a DataCollator
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def format_pair(row):
    # Wrap in chat template
    def fmt(prompt, response):
        msgs = [{"role": "user", "content": prompt},
                {"role": "assistant", "content": response}]
        # Apply template — note: SmolLM2 doesn't officially have system prompt
        return tokenizer.apply_chat_template(msgs, tokenize=False)
    return {
        "chosen": fmt(row["prompt"], row["chosen"]),
        "rejected": fmt(row["prompt"], row["rejected"]),
    }

pref_ds = Dataset.from_list(PREFS).map(format_pair)
print(f"Dataset: {len(pref_ds)} pairs")
print("\nChosen example (truncated):")
print(pref_ds[0]["chosen"][:300])

## 3. Load model & add reward head

In [ ]:
# Cell 4 — Load base model
# We use the BASE model (not instruct) for reward modelling
from transformers import AutoModelForSequenceClassification

# AutoModelForSequenceClassification adds a linear head (num_labels=1 = scalar reward)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=1,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
reward_model.config.pad_token_id = tokenizer.eos_token_id
print("Reward model loaded")
print(f"Output size: {reward_model.config.num_labels}  (scalar reward score)")

## 4. Train with RewardTrainer

In [ ]:
# Cell 5 — Training
reward_config = RewardConfig(
    output_dir="/tmp/reward_model",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=8,
    logging_steps=10,
    bf16=True,
    max_length=256,
    report_to="none",
)

trainer = RewardTrainer(
    model=reward_model,
    args=reward_config,
    train_dataset=pref_ds,
    processing_class=tokenizer,
)

print("Training reward model...")
trainer.train()
print("Done!")

## 5. Score responses

Does the reward model correctly rank better responses above worse ones?

In [ ]:
# Cell 6 — Score test pairs
reward_model.eval()

def get_reward(prompt, response):
    msgs = [{"role": "user", "content": prompt},
            {"role": "assistant", "content": response}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False)
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=256).to(reward_model.device)
    with torch.no_grad():
        score = reward_model(**inputs).logits[0].item()
    return score

TEST_CASES = [
    {
        "prompt": "What is a neural network?",
        "good": "A neural network is a computational model loosely inspired by the brain. It consists of layers of nodes (neurons), each applying a weighted sum followed by a non-linear activation function. Through training on labelled data, it learns weights that map inputs to outputs.",
        "bad": "A neural network is like a brain but made of math. It learns stuff.",
    },
    {
        "prompt": "What is the purpose of a validation set?",
        "good": "A validation set is a held-out portion of data used during training to monitor generalisation. It lets you tune hyperparameters without contaminating the test set. The model never trains on it; you simply evaluate metrics on it periodically to detect overfitting early.",
        "bad": "Validation sets are used to test the model. They are separate from training data.",
    },
]

print(f"{'Prompt':<35} {'Good':>8} {'Bad':>8}  {'Correct?':>8}")
print("-" * 65)
for tc in TEST_CASES:
    g = get_reward(tc["prompt"], tc["good"])
    b = get_reward(tc["prompt"], tc["bad"])
    ok = "✅" if g > b else "❌"
    print(f"{tc['prompt'][:34]:<35} {g:>8.3f} {b:>8.3f}  {ok:>8}")

## ✅ Summary

| Concept | Detail |
|---------|--------|
| **Architecture** | LM backbone + scalar linear head (num_labels=1) |
| **Data** | Preference pairs: (prompt, chosen, rejected) |
| **Loss** | Bradley-Terry: push chosen score above rejected score |
| **Output** | A scalar for any (prompt, response) — higher is better |

**Next**: Module 04 uses this reward model inside an RL loop (PPO) to improve the policy.

> 💡 For production reward models, use a large preference dataset such as:
> ```python
> load_dataset("Anthropic/hh-rlhf")          # 169k pairs
> load_dataset("openai/summarize_from_feedback")
> ```